# 4. Gene Embedding Extraction

This notebook extracts gene embeddings using the validation pipeline (GPU required).

Inference is restricted to the **last three GPUs** (`CUDA_VISIBLE_DEVICES=5,6,7`). Wandb runs in **offline** mode (no API key needed). Check `nvidia-smi` first — those cards must have free memory.

**Outputs** (`.h5ad`, gene-embedding averages) are written to sod2 to avoid filling `/home`:
`/mnt/sod2-project/csb4/stuke1/perturbgen/T_perturb/res/masking`.
Checkpoints are still **read** from `/home/stuke1/perturbgen/T_perturb/res/masking/checkpoints`.

### Recommended: run in `screen` (do not reuse the old home `output_dir` command)

```bash
screen -S gene_emb
bash /home/stuke1/perturbgen/Perturbgen/docs/examples/run_extract_embeddings_sod2.sh
# Ctrl+A then D to detach
# screen -r gene_emb   # reattach later
```

Or re-run this notebook **top to bottom** (params → build cmd → launch) so `OUTPUT_DIR` is sod2.


## 4.1. Parameters (GPU required)

Update the paths and settings for your dataset and checkpoint.


In [8]:
import os
from pathlib import Path

# Workspace root (paths.py ROOT). CLI must be launched with cwd=WORKSPACE.
WORKSPACE = Path("/home/stuke1/perturbgen")
TOKENIZED = WORKSPACE / "T_perturb" / "tokenized_data" / "LPS_all_tps_2k"

# Large outputs go to sod2 (/home is full). Checkpoints stay on home for reading.
SOD2_ROOT = Path("/mnt/sod2-project/csb4/stuke1/perturbgen")

os.environ["CUDA_VISIBLE_DEVICES"] = "5,6,7"
os.environ["WANDB_MODE"] = "offline"
os.environ["WANDB_DIR"] = str(SOD2_ROOT / "wandb")
os.environ["TMPDIR"] = str(SOD2_ROOT / "tmp")

OUTPUT_DIR = str(SOD2_ROOT / "T_perturb" / "res" / "masking")
assert OUTPUT_DIR.startswith("/mnt/sod2-project/csb4"), (
    f"OUTPUT_DIR must be on sod2, got {OUTPUT_DIR}"
)

CKPT_MASKING_PATH = str(
    WORKSPACE
    / "T_perturb"
    / "res"
    / "masking"
    / "checkpoints"
    / "20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt"
)

SRC_DATASET = str(TOKENIZED / "dataset_2000_hvg_src" / "normal.dataset")
TGT_DATASET_FOLDER = str(TOKENIZED / "dataset_2000_hvg_tgt")
SRC_ADATA = str(TOKENIZED / "h5ad_pairing_2000_hvg_src" / "normal.h5ad")
TGT_ADATA_FOLDER = str(TOKENIZED / "h5ad_pairing_2000_hvg_tgt")
MAPPING_DICT_PATH = str(TOKENIZED / "token_id_to_genename_2000_hvg.pkl")
TOKENID_TO_ROWID = str(TOKENIZED / "tokenid_to_rowid_2000_hvg.pkl")

BATCH_SIZE = 64  # model batch size
CELLGEN_LR = 1e-4  # learning rate
CELLGEN_WD = 1e-4  # weight decay
COUNT_LR = 0.001  # learning rate for count head
COUNT_WD = 0.001  # weight decay for count head
D_FF = 32  # feedforward dimension
NUM_LAYERS = 6  # number of transformer layers
N_WORKERS = 4  # workers for data loading (keep modest under DDP)
PRED_TPS = ["1", "2", "3"]  # LPS: "1"=90m, "2"=6h, "3"=10h

VAR_LIST = ["cell_pairing_index", "time_after_LPS", "cell_type_harmonized"]

ENCODER = "scmaskgit"
ENCODER_PATH = str(
    WORKSPACE
    / "Perturbgen"
    / "pretraining_cohort"
    / "20250709_1223_cellgen_train_masking_lr_5e-05_wd_1e-06_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_42-epoch=00.ckpt"
)
CONTEXT_MODE = "True"
MASK_SCHEDULER = "pow"
RETURN_EMBED = "True"
RETURN_ATTN = "False"
GENERATE = "False"
RETURN_GENE_EMBS = "True"
GENE_EMBS_CONDITION = "time_after_LPS"
POS_ENCODING_MODE = "time_pos_sin"
D_MODEL = 768

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(os.environ["WANDB_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["TMPDIR"]).mkdir(parents=True, exist_ok=True)

print("WORKSPACE:", WORKSPACE)
print("OUTPUT_DIR (sod2):", OUTPUT_DIR)
print("WANDB_DIR:", os.environ["WANDB_DIR"])
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("WANDB_MODE:", os.environ["WANDB_MODE"])
print("CKPT:", CKPT_MASKING_PATH)


WORKSPACE: /home/stuke1/perturbgen
CUDA_VISIBLE_DEVICES: 5,6,7
WANDB_MODE: offline
CKPT: /home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt


## 4.2. Build the command

This uses the CLI entrypoint `extract-embedding`, which calls `perturbgen.val`.
Wandb is forced offline; DDP will see only GPUs 5–7 via `CUDA_VISIBLE_DEVICES`.


In [9]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "extract-embedding",
    "--test_mode", "masking",
    "--split", "False",
    "--splitting_mode", "stratified",
    "--return_embed", RETURN_EMBED,
    "--return_attn", RETURN_ATTN,
    "--generate", GENERATE,
    "--ckpt_masking_path", CKPT_MASKING_PATH,
    "--output_dir", OUTPUT_DIR,
    "--src_dataset", SRC_DATASET,
    "--tgt_dataset_folder", TGT_DATASET_FOLDER,
    "--src_adata", SRC_ADATA,
    "--tgt_adata_folder", TGT_ADATA_FOLDER,
    "--mapping_dict_path", MAPPING_DICT_PATH,
    "--batch_size", str(BATCH_SIZE),
    "--cellgen_lr", str(CELLGEN_LR),
    "--cellgen_wd", str(CELLGEN_WD),
    "--count_lr", str(COUNT_LR),
    "--count_wd", str(COUNT_WD),
    "--d_ff", str(D_FF),
    "--num_layers", str(NUM_LAYERS),
    "--n_workers", str(N_WORKERS),
    "--pred_tps", *PRED_TPS,
    "--var_list", *VAR_LIST,
    "--tokenid_to_rowid", TOKENID_TO_ROWID,
    "--encoder", ENCODER,
    "--encoder_path", ENCODER_PATH,
    "--context_mode", CONTEXT_MODE,
    "--mask_scheduler", MASK_SCHEDULER,
    "--return_gene_embs", RETURN_GENE_EMBS,
    "--gene_embs_condition", GENE_EMBS_CONDITION,
    "--pos_encoding_mode", POS_ENCODING_MODE,
    "--d_model", str(D_MODEL),
    "--wandb_mode", "offline",
]

print(" ".join(cmd))


python -m perturbgen extract-embedding --test_mode masking --split False --splitting_mode stratified --return_embed True --return_attn False --generate False --ckpt_masking_path /home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt --output_dir /home/stuke1/perturbgen/T_perturb/res/masking --src_dataset /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_2000_hvg.pkl --batch_size 64 --cellgen_lr 0.0

In [10]:
import os
import subprocess

assert "sod2-project" in OUTPUT_DIR, (
    f"Refusing to run: OUTPUT_DIR is not on sod2: {OUTPUT_DIR}\n"
    "Re-run the parameters cell first."
)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "5,6,7"
env["WANDB_MODE"] = "offline"
env["WANDB_DIR"] = str(SOD2_ROOT / "wandb")
env["TMPDIR"] = str(SOD2_ROOT / "tmp")

print("Launching extract-embedding on GPUs", env["CUDA_VISIBLE_DEVICES"])
print("OUTPUT_DIR:", OUTPUT_DIR)
print(" ".join(cmd))

subprocess.run(cmd, check=True, cwd=str(WORKSPACE), env=env)


Launching extract-embedding on GPUs 5,6,7
python -m perturbgen extract-embedding --test_mode masking --split False --splitting_mode stratified --return_embed True --return_attn False --generate False --ckpt_masking_path /home/stuke1/perturbgen/T_perturb/res/masking/checkpoints/20260729_1751_cellgen_train_masking_lr_0.0001_wd_0.0001_batch_64_ptime_pos_sin_m_pow_tp_1-2-3_s_0-epoch=19.ckpt --output_dir /home/stuke1/perturbgen/T_perturb/res/masking --src_dataset /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_src/normal.dataset --tgt_dataset_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/dataset_2000_hvg_tgt --src_adata /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_src/normal.h5ad --tgt_adata_folder /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/h5ad_pairing_2000_hvg_tgt --mapping_dict_path /home/stuke1/perturbgen/T_perturb/tokenized_data/LPS_all_tps_2k/token_id_to_genename_200

Seed set to 42
/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 2_6h_LPS.h5ad...


/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading 1_90m_LPS.h5ad...


/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Exception ignored in: <function _xla_gc_callback at 0x7fad2629b7e0>
Traceback (most recent call last):
  File "/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 


KeyboardInterrupt: 